In [1]:
# --- output directories (created relative to repo root) ---
from pathlib import Path as _P
for _d in ('figures','figures/figure2','figures/figure3','figures/figure4','figures/figure5'):
    _P(_d).mkdir(parents=True, exist_ok=True)

import sys, os
sys.path.insert(0, os.path.abspath("src"))
import re
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import curve_fit
import importlib, core
importlib.reload(core)
from core import count_sites_per_sample_ptm_report

In [2]:
# Palettes (match Figure 4): FF = red, FFPE = violet
FF_COLOR   = '#e64126'
FFPE_COLOR = '#846db1'
INPUTS = [10, 20, 50, 100, 200, 500, 1000]   # ng protein input (dilution series)

# Data upload

In [3]:
# FF / FFPE mouse-brain dilution-series PTM Site Reports (Class I per-run, 10 ng - 1 ug, n=3).
RAW_DIR = Path('pride_data/analysis_data/revision/figure4')

def find_report(material, ng):
    if material == 'FF' and ng == 1000:
        return next(RAW_DIR.glob('*dilser_FF_1000ng_repeat_Report.tsv'))   # FF 1 ug = repeat acquisition
    return next(RAW_DIR.glob(f'*dilser_{material}_{ng}ng_Report.tsv'))

# per-replicate Class I phosphosite counts (multiplicity collapsed, localization enforced)
ff_counts   = {ng: list(count_sites_per_sample_ptm_report(pd.read_csv(find_report('FF', ng),   sep='\t', low_memory=False)).values()) for ng in INPUTS}
ffpe_counts = {ng: list(count_sites_per_sample_ptm_report(pd.read_csv(find_report('FFPE', ng), sep='\t', low_memory=False)).values()) for ng in INPUTS}
print(pd.DataFrame({'FF_mean':[int(np.mean(ff_counts[n])) for n in INPUTS],
                    'FFPE_mean':[int(np.mean(ffpe_counts[n])) for n in INPUTS]}, index=INPUTS).to_string())

      FF_mean  FFPE_mean
10        453        126
20        921        227
50       1646        466
100      3303       1078
200      5086       2229
500      7878       3507
1000     8489       7283


# Supplementary Figure 4a

FF/FFPE phosphopeptide-precursor depth across the protein-input dilution series (mouse brain) — the precursor-level mirror of Fig. 4a (Class I sites), same grouped-box convention and FF/FFPE colours. Phosphopeptide precursors = unique phosphorylated precursors (`EG.PrecursorId`), localization-independent, per Bekker-Jensen et al. 2020 (n = 3 per input).

In [4]:
# Supplementary Figure 4a — FF/FFPE phosphopeptide-precursor depth vs input (precursor-level mirror of Fig. 4a).
# Phosphopeptide precursors = unique phosphorylated precursors (EG.PrecursorId), localization-independent,
# per Bekker-Jensen 2020. Counted per RUN, decoys removed (n=3 per input). Grouped box, as Fig. 4a.
import os, glob
import numpy as np, pandas as pd
import plotly.graph_objects as go
from core import _hex_to_rgba

PREC_DIR = r'pride_data/analysis_data/figure4'
INPUTS = [10, 20, 50, 100, 200, 500, 1000]
FF_COLOR, FFPE_COLOR = '#e64126', '#846db1'      # match Figure 4

def count_phosphoprecursors_per_run(path):
    """Unique phosphorylated precursors (EG.PrecursorId) per run; decoys removed."""
    df = pd.read_csv(path, sep='\t',
                     usecols=['R.FileName', 'EG.ModifiedSequence', 'EG.PrecursorId', 'EG.IsDecoy'],
                     low_memory=False)
    decoy = (df['EG.IsDecoy'].astype(str).str.lower().isin(['true', '1'])
             if df['EG.IsDecoy'].dtype != bool else df['EG.IsDecoy'])
    df = df[~decoy]
    df = df[df['EG.ModifiedSequence'].astype(str).str.contains('Phospho', case=False, na=False)]
    return df.groupby('R.FileName')['EG.PrecursorId'].nunique().tolist()

def find_prec(material, ng):
    hits = glob.glob(os.path.join(PREC_DIR, f'*dilser_{material}_{ng}ng_Report.tsv'))
    return hits[0] if hits else None

ff_prec_counts, ffpe_prec_counts = {}, {}
for ng in INPUTS:
    fp, pp = find_prec('FF', ng), find_prec('FFPE', ng)
    if fp: ff_prec_counts[ng] = count_phosphoprecursors_per_run(fp)
    if pp: ffpe_prec_counts[ng] = count_phosphoprecursors_per_run(pp)

rows = [ng for ng in INPUTS if ng in ff_prec_counts and ng in ffpe_prec_counts]
summary = pd.DataFrame({
    'FF_mean':   [int(np.mean(ff_prec_counts[ng]))   for ng in rows],
    'FFPE_mean': [int(np.mean(ffpe_prec_counts[ng])) for ng in rows],
}, index=rows)
summary['FF/FFPE'] = (summary['FF_mean'] / summary['FFPE_mean']).round(2)
summary.index.name = 'input_ng'
print(summary.to_string())

xcat = [str(n) for n in INPUTS]
fig = go.Figure()
for label, counts, color in [('Fresh-frozen', ff_prec_counts, FF_COLOR), ('FFPE', ffpe_prec_counts, FFPE_COLOR)]:
    fig.add_trace(go.Box(
        y=[v for ng in INPUTS if ng in counts for v in counts[ng]],
        x=[str(ng) for ng in INPUTS if ng in counts for _ in counts[ng]],
        name=label, boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=7, color=color, line=dict(width=0.5, color='black')),
        line=dict(color=color, width=1.5), fillcolor=_hex_to_rgba(color, 0.15)))
fig.update_layout(width=600, height=600, template='plotly_white', boxmode='group',
                  xaxis_title='Protein input (ng)', yaxis_title='Phosphopeptide precursors', showlegend=False)
fig.update_xaxes(categoryorder='array', categoryarray=xcat)
fig.update_yaxes(range = [0,50500])
fig.show()
#fig.write_image(r'figures/figure4/suppl_figure4a.pdf', width=600, height=600)


          FF_mean  FFPE_mean  FF/FFPE
input_ng                             
10           1904        534     3.57
20           4509        971     4.64
50           8172       1820     4.49
100         17454       4948     3.53
200         25754       9938     2.59
500         40832      16454     2.48
1000        45195      33857     1.33


# Supplementary Figure 4b & 4c
Class I phosphosite depth vs protein input for fresh-frozen (3a) and FFPE (3b) mouse brain, each fitted with the **same Michaelis–Menten saturation model** `sites = Vmax·x/(Km+x)`. Using one consistent model for both materials makes the difference interpretable from the fitted parameters (Km, % of Vmax reached) rather than from the choice of equation. Fresh-frozen approaches saturation within the series (low Km), whereas FFPE remains far below its plateau (high Km) — indicating FFPE depth is input-limited and that phosphopeptide recovery from FFPE is lower at matched input.

In [5]:
# Suppl 4b/3c - Michaelis-Menten fit (same model both materials), fitted to all replicate points.
def michaelis_menten(x, Vmax, Km):
    return Vmax * x / (Km + x)

def r_squared(y, yhat):
    ss_res = np.sum((y - yhat) ** 2); ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1 - ss_res / ss_tot

def mm_panel(counts, color, title):
    x = np.array([ng for ng in INPUTS for _ in counts[ng]], dtype=float)
    y = np.array([v for ng in INPUTS for v in counts[ng]], dtype=float)
    (Vmax, Km), _ = curve_fit(michaelis_menten, x, y, p0=[max(y) * 1.5, 200], maxfev=10000)
    r2 = r_squared(y, michaelis_menten(x, Vmax, Km))
    pct_1ug = michaelis_menten(1000, Vmax, Km) / Vmax * 100
    print(f'{title:<12} Vmax={Vmax:7.0f}  Km={Km:7.0f} ng  R2={r2:.4f}  | {pct_1ug:.0f}% of Vmax at 1 ug')

    xs = np.linspace(min(INPUTS), max(INPUTS), 300)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers',
        marker=dict(size=11, color=color, line=dict(width=0.5, color='black')), showlegend=False))
    fig.add_trace(go.Scatter(x=xs, y=michaelis_menten(xs, Vmax, Km), mode='lines',
        line=dict(color=color, width=3, dash='dash'), showlegend=False))
    fig.add_annotation(x=0.04, y=0.96, xref='paper', yref='paper', showarrow=False, align='left',
        text=(f'Vmax = {Vmax:,.0f}<br>Km = {Km:,.0f} ng<br>'
              f'{pct_1ug:.0f}% of Vmax at 1 ug<br>R&#178; = {r2:.3f}'))
    fig.update_layout(width=600, height=600, template='plotly_white', title=title,
                      xaxis_title='Protein input (ng)', yaxis_title='Class I phosphosites')
    fig.update_yaxes(rangemode='tozero')
    return fig, (Vmax, Km, r2, pct_1ug)

fig_a, ff_fit   = mm_panel(ff_counts,   FF_COLOR,   'Fresh-frozen (3b)')
fig_b, ffpe_fit = mm_panel(ffpe_counts, FFPE_COLOR, 'FFPE (3c)')
fig_a.show()
fig_b.show()
#fig_a.write_image(r'figures/figure4/suppl_figure4b.pdf', width=600, height=600)
#fig_b.write_image(r'figures/figure4/suppl_figure4c.pdf', width=600, height=600)

Fresh-frozen (3b) Vmax=  10749  Km=    223 ng  R2=0.9932  | 82% of Vmax at 1 ug
FFPE (3c)    Vmax=  35294  Km=   3924 ng  R2=0.9853  | 20% of Vmax at 1 ug


# Supplementary Figure 4d
Fresh-frozen / FFPE Class I phosphosite-depth ratio (ratio of per-input means) across the dilution series. The advantage of fresh-frozen over FFPE is largest at low input and narrows toward 1 µg.

In [6]:
# Suppl 4d - FF / FFPE Class I depth ratio per input (ratio of means).
ratio = [np.mean(ff_counts[ng]) / np.mean(ffpe_counts[ng]) for ng in INPUTS]
print(pd.DataFrame({'FF/FFPE': np.round(ratio, 2)}, index=INPUTS).to_string())

fig = go.Figure()
fig.add_trace(go.Scatter(x=[str(n) for n in INPUTS], y=ratio, mode='lines+markers',
    marker=dict(size=13, color='#312353'), line=dict(color='#846db1', width=2)))
fig.add_hline(y=1.0, line=dict(color='black', dash='dot', width=1.5))
fig.update_layout(width=600, height=600, template='plotly_white',
                  xaxis_title='Protein input (ng)', yaxis_title='Fresh-frozen / FFPE Class I ratio')
fig.update_yaxes(rangemode='tozero')
fig.show()
#fig.write_image(r'figures/figure4/suppl_figure4d.pdf', width=600, height=600)

      FF/FFPE
10       3.59
20       4.06
50       3.53
100      3.06
200      2.28
500      2.25
1000     1.17


In [7]:
# === PRIDE MetaInfo export (run after all panels above) ===
import sys; sys.path.insert(0, r"src")
from metainfo_export import dump_panel
SFIG = 4   # figure number (single source of truth for sheet labels)
from core import count_sites_per_sample_ptm_report, process_ptm_site_report
import numpy as np, pandas as pd
def _try(fn, sheet):
    try: fn()
    except Exception as e: print(f"  [SKIP {sheet}] {type(e).__name__}: {e}")

_try(lambda: dump_panel(pd.DataFrame([{"Material":mat,"Input_ng":ng,"Replicate":i+1,"Phosphopeptide precursors":int(v)}
    for mat,dct in [("FF",ff_prec_counts),("FFPE",ffpe_prec_counts)] for ng in sorted(dct) for i,v in enumerate(dct[ng])]),f"Suppl Figure {SFIG}a"),f"Suppl Figure {SFIG}a")

def _s3ab():
    rows=[]
    for ng in INPUTS:
        for v in ff_counts[ng]:   rows.append({"Material":"FF","Input_ng":ng,"Number of class I sites":int(v)})
        for v in ffpe_counts[ng]: rows.append({"Material":"FFPE","Input_ng":ng,"Number of class I sites":int(v)})
    dump_panel(pd.DataFrame(rows),f"Suppl Figure {SFIG}b, {SFIG}c")
_try(_s3ab,f"Suppl Figure {SFIG}b, {SFIG}c")
_try(lambda: dump_panel(pd.DataFrame({"Input_ng":INPUTS,
    "FF_mean":[np.mean(ff_counts[ng]) for ng in INPUTS],
    "FFPE_mean":[np.mean(ffpe_counts[ng]) for ng in INPUTS],
    "Ratio":[np.mean(ff_counts[ng])/np.mean(ffpe_counts[ng]) for ng in INPUTS]}),
    f"Suppl Figure {SFIG}d"),f"Suppl Figure {SFIG}d")
print(f"Suppl Figure {SFIG} export done.")


  [MetaInfo] wrote 'Suppl Figure 4a'  (42 rows x 4 cols)
  [MetaInfo] wrote 'Suppl Figure 4b, 4c'  (42 rows x 3 cols)
  [MetaInfo] wrote 'Suppl Figure 4d'  (7 rows x 4 cols)
Suppl Figure 4 export done.
